# 01. Download Dataset

In [ ]:
from datasets import load_dataset
import pandas as pd
import os

In [ ]:
dataset_name = 'bitext/Bitext-customer-support-llm-chatbot-training-dataset'
dataset = load_dataset(dataset_name, split='train')

In [ ]:
df = dataset.to_pandas()
print(f'Loaded {len(df)} rows')
df.head()

In [ ]:
os.makedirs('data/raw', exist_ok=True)
df.to_json('data/raw/dataset.jsonl', orient='records', lines=True)

# 02. Data Exploratio

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_json('data/raw/dataset.jsonl', lines=True)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

# 03. Preprocessing

In [ ]:
import pandas as pd
import os

In [ ]:
df = pd.read_json('data/raw/dataset.jsonl', lines=True)

In [ ]:
# Drop nulls and duplicates
df = df.dropna(subset=['instruction', 'response'])
df = df.drop_duplicates(subset=['instruction', 'response'])
print(f'Cleaned dataset has {len(df)} rows')

In [ ]:
os.makedirs('data/processed', exist_ok=True)
df.to_json('data/processed/cleaned_pairs.jsonl', orient='records', lines=True)

# 04. Prompt Formatting & Splitting

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os

In [ ]:
df = pd.read_json('data/processed/cleaned_pairs.jsonl', lines=True)

In [ ]:
def format_prompt(instruction, response):
    return f'''### Instruction:\n
Customer:\n
{instruction.strip()}\n
\n
### Response:\n
{response.strip()}'''

In [ ]:
df['text'] = df.apply(lambda row: format_prompt(row['instruction'], row['response']), axis=1)
print(df['text'].iloc[0])

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(f'Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')

In [ ]:
os.makedirs('data/processed', exist_ok=True)
train_df.to_json('data/processed/train.jsonl', orient='records', lines=True)
val_df.to_json('data/processed/val.jsonl', orient='records', lines=True)
test_df.to_json('data/processed/test.jsonl', orient='records', lines=True)

# 05. Fine-tune LLM

In [ ]:
# !pip install 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'

In [ ]:
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import os

In [ ]:
max_seq_length = 2048
base_model_name = 'unsloth/Llama-3.2-3B-Instruct-bnb-4bit'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
)

In [ ]:
train_dataset = load_dataset('json', data_files='data/processed/train.jsonl', split='train')

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field='text',
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='linear',
        seed=42,
        output_dir='outputs',
    ),
)

In [ ]:
trainer.train()

In [ ]:
model.save_pretrained('models/lora_adapter')
tokenizer.save_pretrained('models/lora_adapter')

# 06. Evaluatio

In [ ]:
from unsloth import FastLanguageModel
import pandas as pd

In [ ]:
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='models/lora_adapter',
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

In [ ]:
test_df = pd.read_json('data/processed/test.jsonl', lines=True)

def generate_response(instruction):
    prompt = f'''### Instruction:\n
Customer:\n
{instruction.strip()}\n
\n
### Response:\n'''
    inputs = tokenizer([prompt], return_tensors='pt').to('cuda')
    outputs = model.generate(**inputs, max_new_tokens=256, use_cache=True)
    response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    return response.split('### Response:\n')[-1]

In [ ]:
sample = test_df.iloc[0]
print('Instruction:', sample['instruction'])
print('\nActual Response:', sample['response'])
print('\nGenerated Response:', generate_response(sample['instruction']))

# 07. Inference

In [ ]:
from unsloth import FastLanguageModel

In [ ]:
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='models/lora_adapter',
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

In [ ]:
def chat_with_bot(instruction):
    prompt = f'''### Instruction:\n
Customer:\n
{instruction.strip()}\n
\n
### Response:\n'''
    inputs = tokenizer([prompt], return_tensors='pt').to('cuda')
    outputs = model.generate(**inputs, max_new_tokens=256, use_cache=True)
    response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    return response.split('### Response:\n')[-1]

In [ ]:
print(chat_with_bot('I need help tracking my order #12345.'))